In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import tensorflow as tf

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import spacy

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression 
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder
from scipy.sparse import hstack, csr_matrix
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Dropout, Concatenate
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, mean_absolute_error, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
import lightgbm as lgb
from scipy.stats import spearmanr

In [ ]:
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
df_train_clean = pd.read_csv("/kaggle/input/notebooks/irakosenkoiryna389/final-project-eda-kosenko/df_train_clean.csv")
df_train_clean.shape

In [ ]:
df_train_clean.info()

In [ ]:
df_test_clean = pd.read_csv("/kaggle/input/notebooks/irakosenkoiryna389/final-project-eda-kosenko/df_test_clean.csv")
df_test_clean.shape


In [ ]:
df_test_clean.info()

# Demonstration of applying classical NLP techniques to the text data, along with an explanation of how they were used in the model (if they were used) or why they weren't used (if they weren't)


**Класичні NLP-техніки**

1. Tokenization — розбиття тексту на слова/токени
2. Lowercasing  — приведення до нижнього регістру
3. Stop words removal  — видалення службових слів (the, a, is...)
4. Stemming — обрізання закінчень (running → run)
5. Lemmatization — приведення до базової форми (better → good). 
6. POS tagging  — визначення частин мови
7. TF-IDF — числове представлення тексту
8. N-grams — комбінації сусідніх слів

Лематизація обрана замість стемінгу, тому що відгуки на одяг насичені прикметниками. Стемінг-спотворює, лематизація зберігає словникову форму.

Стоп-слова видаляємо тільки для BoW, де важлива частота слів, а не порядок. Для RNN їх видалення руйнує граматичний контекст.

In [ ]:
# Завантаження мовної моделі spaCy

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])  
#  вимикаємо синтаксичний парсер та NER (розпізнавання іменованих сутностей),
# бо вони не потрібні для BoW і лише уповільнюють обробку

In [ ]:
# лематизація: love/loved/loving -> love
# видалення стоп-слів: the, is, a, I..., 
# видалення пунктуації та пробілів
# видалення дуже коротких токенів (1-2 символи)

def clean_text(text):
    doc = nlp(str(text))  # doc - це не просто рядок, а послідовність токенів, де кожен токен вже проаналізований
                            # token.text, token.lemma_, token.is_stop, token.is_punct
    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop      
        and not token.is_punct    
        and not token.is_space    
        and len(token.lemma_) > 2   
    ]
    return " ".join(tokens)

In [ ]:
# Застосування очищення до текстів

df_train_clean["clean_text"] = df_train_clean["Full_Review"].apply(clean_text)
df_test_clean["clean_text"] = df_test_clean["Full_Review"].apply(clean_text)

# Перевірка результату
print("\nПриклад оригінального тексту:")
# print(df_train_clean["Full_Review"].iloc[0])
print(df_train_clean["Full_Review"].iloc[5])
print("\nПісля обробки spaCy:")
# print(df_train_clean["clean_text"].iloc[0])
print(df_train_clean["clean_text"].iloc[5])

In [ ]:
# Виділяємо цільові змінні (таргети)
y_recommended = df_train_clean["Recommended"]
y_rating = df_train_clean["Rating"]

# Створюємо чисту матрицю ознак X, видаляючи обидва таргет-стовпчики
X = df_train_clean.drop(columns=["Recommended", "Rating"])

# Робимо правильний спліт для Recommended
# Розбиваю на тренувальні та валідаційні дані для того, щоб можна було виміряти точність передбачення без submission
X_train, X_val, y_recommended_train, y_recommended_val = train_test_split(
    X,
    y_recommended, 
    test_size=0.2, 
    random_state=8, 
    stratify=y_recommended
)

# Робимо синхронний спліт для Rating за тими ж індексами (за допомогою random_state=8)
_, _, y_rating_train, y_rating_val = train_test_split(
    X, 
    y_rating, 
    test_size=0.2, 
    random_state=8, 
    stratify=y_recommended
)

y_recommended.shape, y_recommended_train.shape, y_recommended_val.shape, y_rating.shape, y_rating_train.shape, y_rating_val.shape

# Demonstration of a Bag-of-Words model for the required task

fit: CountVectorizer проходить по всіх текстах і будує словник — перелік унікальних токенів з присвоєним індексом.

transform: Кожен текст перетворюється на вектор довжиною len(словника). Кожна позиція — кількість входжень відповідного слова.

In [ ]:
# Тепер застосуємо Bag-of-Words: рахуємо скільки разів кожне слово/біграма зустрічається в тексті

# bow = CountVectorizer(
#     # max_features=5000,       # беремо лише 5000 найчастіших токенів
#     ngram_range=(1, 2),      # враховуємо окремі слова (1-gram) та пари слів (2-gram)
#     min_df=5)                # ігноруємо токени, що зустрічаються рідше ніж у 5 документах            

# X_bow_train = bow.fit_transform(df_train_clean["clean_text"]) 
# X_bow_val = bow.transform(df_val["clean_text"])  # для оцінки без submission
# X_bow_test = bow.transform(df_test_clean["clean_text"])     

# # Матриця розміром [кількість текстів × розмір словника].

# print("train:", X_bow_train.shape)
# print("test:", X_bow_test.shape)
# print("valid:", X_bow_val.shape)

# print(X_bow_train)

# # Приклад найчастіших слів у словнику
# vocab = bow.get_feature_names_out()
# print("\n10 токенів у словнику:", vocab[:10])

In [ ]:
 # Заміна CountVectorizer на TfidfVectorizer

# min_df=5 ігнорує слова, які зустрічаються менше ніж у 5 відгуках
# max_df=0.95 ігнорує слова, які є у понад 95% відгуків
tfidf = TfidfVectorizer(min_df=5, max_df=0.95, ngram_range=(1, 2))

# fit_transform робимо виключно на Train, на Val та Test тільки transform
X_tfidf_train = tfidf.fit_transform(X_train["clean_text"])
X_tfidf_val = tfidf.transform(X_val["clean_text"])
X_tfidf_test = tfidf.transform(df_test_clean["clean_text"])

print(f"Розмірність TF-IDF матриці: {X_tfidf_train.shape}")
print(f"Фактичний розмір словника (кількість ознак): {X_tfidf_train.shape[1]}")

In [ ]:
# Обробка категоріальних ознак за допомогою OneHotEncoder
# Пайплайн налаштовано так, щоб ігнорувати нові категорії

cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)

cat_cols = ["Division", "Department", "Product_Category"]
X_cat_train = cat_encoder.fit_transform(X_train[cat_cols])
X_cat_test= cat_encoder.transform(df_test_clean[cat_cols])
X_cat_val = cat_encoder.transform(X_val[cat_cols])

# Стандартизація числових ознак
scaler = StandardScaler()
num_cols = ["Age", "Pos_Feedback_Cnt"]

# Навчання скалера тільки на train для запобігання Data Leakage
X_num_train = scaler.fit_transform(X_train[num_cols].values)
X_num_test = scaler.transform(df_test_clean[num_cols].values)
X_num_val = scaler.transform(X_val[num_cols].values)

# Об'єднання ознак (Конкатенація матриць)
# Оскільки OneHotEncoder повертає розріджену матрицю (sparse matrix), 
# замість np.hstack ефективніше використовувати scipy.sparse.hstack.
X_extra_train = hstack([csr_matrix(X_num_train), X_cat_train]).tocsr()
X_extra_test = hstack([csr_matrix(X_num_test),  X_cat_test]).tocsr()
X_extra_val = hstack([csr_matrix(X_num_val),   X_cat_val]).tocsr()

print("Розмір train:", X_extra_train.shape)
print("Перший рядок матриці (у розрідженому форматі):")
print(X_extra_train[0])

In [ ]:
# Горизонтальне об'єднання: TfidfVectorizer + числові + категоріальні

X_train = hstack([X_tfidf_train, X_extra_train])
X_test = hstack([X_tfidf_test,  X_extra_test])
X_val = hstack([X_tfidf_val,  X_extra_val])

print("Final train:", X_train.shape)
print("Final test: ", X_test.shape)
print("Final validation: ", X_val.shape)

In [ ]:
# Модель для Recommended

# Recommended — це бінарна задача (0 або 1). 
# Метрика F1 є більш інформативною за accuracy, коли є дисбаланс класів (у нашому датасеті більшість відгуків позитивні). 
# solver="saga" оптимальний для великих розріджених матриць.

model_rec = LogisticRegression(max_iter=2000, C=1.0, solver="saga", n_jobs=-1, class_weight="balanced")
#  збільшувала кількість ітерацій тому що було попередження ConvergenceWarning

# Оцінюємо через 5-fold крос-валідацію
scores_rec = cross_val_score(model_rec, X_train, y_recommended_train, cv=5, scoring="f1_macro", n_jobs=-1)

print(f"Recommended — F1_macro(крос-валідація): {scores_rec.mean():.3f} ± {scores_rec.std():.3f}")

# # Навчаємо на всьому train
# model_rec.fit(X_train, y_recommended)

In [ ]:
# Інша модель для Recommended, спробуємо LGBMClassifier

model_rec_lgb = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05,
                                   random_state=8, n_jobs=-1, class_weight="balanced")

model_rec_lgb.fit(X_train, y_recommended_train, eval_set=[(X_val, y_recommended_val)],
                                                eval_metric="binary_logloss",
                                                callbacks=[lgb.early_stopping(stopping_rounds=50)])

In [ ]:
best_n_estimators_rec = model_rec_lgb.best_iteration_
print("Оптимальна кількість дерев:", best_n_estimators_rec)

In [ ]:
# Модель для Rating (мультикласова класифікація)

# model_rating = LogisticRegression(max_iter=2000, C=1.0, solver="saga",n_jobs=-1, class_weight="balanced")

# model_rating.fit(X_train, y_rating)

In [ ]:
# Змінюю на іншу модель LGBMRegressor, як для регресії

model_rating_lgb = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    objective='regression',
    random_state=8
)
# Навчаємо на числових мітках рейтингу (у стовпчику мають бути числа 1, 2, 3, 4, 5)

model_rating_lgb.fit(X_train, y_rating_train)

In [ ]:
# Для оцінки точності на валідаційних даних 

# Ймовірності класу 1, а не готові класи
proba_val_lgb = model_rec_lgb.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.1, 0.91, 0.05)
f1_macro_scores = [
    f1_score(y_recommended_val, (proba_val_lgb >= t).astype(int), average="macro")
    for t in thresholds
]

best_threshold = thresholds[np.argmax(f1_macro_scores)]
print(f"Найкращий поріг: {best_threshold:.2f}, F1 macro: {max(f1_macro_scores):.3f}")

pred_recommended_val_lgb = (proba_val_lgb >= best_threshold).astype(int)

print("Recommended")
print(classification_report(y_recommended_val, pred_recommended_val_lgb))


pred_rating_val_lgb = model_rating_lgb.predict(X_val) 
print("Rating")
print(f"MAE: {mean_absolute_error(y_rating_val, pred_rating_val_lgb):.3f}")

# Рахуємо коефіцієнт Спірмена
spearman_rating_lgb = spearmanr(y_rating_val, pred_rating_val_lgb).statistic
print(f"Коефіцієнт Спірмена: {spearman_rating_lgb}")

In [ ]:
from scipy.sparse import vstack

# Тренування моделей на повному датасеті, об'єднати тренувальні та валідаційні дані

# Об'єднання ознак: вертикальна конкатенація розріджених матриць
X_full_train = vstack([X_train, X_val]).tocsr()

# Об'єднання таргетів: порядок рядків у y має збігатися з порядком у X_full_train
y_recommended_full_train = pd.concat([y_recommended_train, y_recommended_val], ignore_index=True)
y_rating_full_train = pd.concat([y_rating_train, y_rating_val], ignore_index=True)

print("X_full_train:", X_full_train.shape)
print("y_recommended_full_train:", y_recommended_full_train.shape)
print("y_rating_full_train:", y_rating_full_train.shape)

In [ ]:
# Фінальні моделі для сабміту — нові імена, щоб не перезаписати
# model_rec_lgb / model_rating_lgb, які оцінені на X_val у клітинці 24

model_rec_lgb_final = lgb.LGBMClassifier(
    n_estimators=best_n_estimators_rec,
    learning_rate=0.05,
    random_state=8,
    n_jobs=-1,
    class_weight="balanced",
)
model_rec_lgb_final.fit(X_full_train, y_recommended_full_train)

model_rating_lgb_final = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    objective="regression",
    random_state=8,
)
model_rating_lgb_final.fit(X_full_train, y_rating_full_train)

In [ ]:
# Передбачення на тесті фінальними повними моделями

proba_test_lgb = model_rec_lgb_final.predict_proba(X_test)[:, 1]
pred_recommended_lgb = (proba_test_lgb >= best_threshold).astype(int)

pred_rating_lgb = model_rating_lgb_final.predict(X_test)

In [ ]:
# Передбачення на тестових даних

# pred_rating_lgb = model_rating_lgb.predict(X_test)

# proba_test_lgb = model_rec_lgb.predict_proba(X_test)[:, 1]
# pred_recommended_lgb = (proba_test_lgb >= best_threshold).astype(int)

In [ ]:
# Підготовка файлу до submission 

submission = pd.DataFrame({
    "Id": df_test_clean["Id"].values,
    "Rating": pred_rating_lgb,
    "Recommended": pred_recommended_lgb
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

print("Submission збережено.")
print(submission.head(10))
print("\nРозподіл передбачених Rating:")
print(submission["Rating"].value_counts().sort_index())
print("\nРозподіл передбачених Recommended:")
print(submission["Recommended"].value_counts())

# Demonstration of an RNN-based model for the required task

**Є такі моделі:**

Simple RNN (Vanilla RNN)

LSTM (Long Short-Term Memory)

GRU (Gated Recurrent Unit)

Bidirectional RNN (Двонаправлені

Deep RNN (Глибокі).

**Використаю BiLSTM**

Звичайний LSTM читає текст лише в одному напрямку — зліва направо. 

Bidirectional LSTM запускає два LSTM паралельно:перший читає зліва направо, другий читає справа наліво

In [ ]:
# RNN моделі потребують трохи іншої очистки тексту.
# У BoW-підході порядок слів втрачається, тому там видалялися стоп-слова. 
# У RNN порядок слів — це і є основна інформація. Тому для RNN стоп-слова не видаляємо, 
# лематизацію можна зробити (зменшує словник без втрати змісту),
# мінімальна очистка: нижній регістр, видалення спецсимволів

def clean_text_rnn(text):
    doc = nlp(str(text))
    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_punct
        and not token.is_space
    ]
    return " ".join(tokens)

In [ ]:
df_train_clean["rnn_text"] = df_train_clean["Full_Review"].apply(clean_text_rnn)
df_test_clean["rnn_text"]  = df_test_clean["Full_Review"].apply(clean_text_rnn)

# ттреба перевірити як виглядає відгук після очистки
print(df_train_clean["Full_Review"].iloc[5])
print("->", df_train_clean["rnn_text"].iloc[5])

In [ ]:
# Keras-токенізатор будує словник і перетворює кожне слово на ціле число.

# MAX_WORDS = 15000         # розмір словника: 15 000 найчастіших слів
MAX_WORDS = 20000         # покращення: спробую більший розмір словника
# MAX_LEN_REVIEW = 150     # максимальна довжина відгуку
MAX_LEN_REVIEW = 108      # покращення: треба перевірити довжину 

# Навчаємо токенізатор тільки на тренувальних даних
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>") # розбиває суцільний текст на окремі мінімальні смислові одиниці, що називаються токенами
tokenizer.fit_on_texts(df_train_clean["rnn_text"])

# Перетворення текстів у послідовності індексів
X_train_seq = tokenizer.texts_to_sequences(df_train_clean["rnn_text"])
X_test_seq  = tokenizer.texts_to_sequences(df_test_clean["rnn_text"])
print(X_train_seq[5])

# Вирівнювання довжини: коротші — доповнюємо нулями, довші — обрізаємо
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN_REVIEW, padding="post", truncating="post")
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN_REVIEW, padding="post", truncating="post")
print(X_train_pad[5])

print("Train shape:", X_train_pad.shape)  
print("Test shape: ", X_test_pad.shape)

In [ ]:
# Покращення:
# Перевірити оптимальність MAX_LEN_REVIEW, якщо менше чим 150, то краще використовувати інше значення

lengths = [len(s) for s in X_train_seq]
print(np.percentile(lengths, [50, 75, 90, 95])) # треба взяти значення 95-го перцентилю

In [ ]:
# Підготовка числових та категоріальних ознак

# категоріальні ознаки можна взяти із попередньої підготовки: X_cat_train та X_cat_test 

# Числові ознаки треба нормалізувати, так як це важливо для нейронних мереж
# використовую StandardScaler замість MinMaxScaler тому що є значні викиди 
scaler = StandardScaler()
num_cols = ["Age", "Pos_Feedback_Cnt"]

X_num_train = scaler.fit_transform(df_train_clean[num_cols])
X_num_test  = scaler.transform(df_test_clean[num_cols])

# Об'єднуємо в один dense-масив
X_total_train = np.hstack([X_num_train, X_cat_train]) 
X_total_test  = np.hstack([X_num_test,  X_cat_test])

X_total_train.shape

In [ ]:
# Розбиваємо на тренувальн та валідаційні дані

# після розділення на тренувальні та валідаційні дані для ML - оцінка суттєво зменшилася
# можливо, через зменшення кількості тренувальних даних.
# Їх недостатньо для нейронних мереж. Може це одна із причин низької оцінки.

(X_text_train, X_text_val,
 X_extra_train, X_extra_val,
 y_rat_train, y_rat_val,
 y_rec_train,y_rec_val) = train_test_split(X_train_pad, X_total_train, 
                                           y_rating, y_recommended,
                                           test_size=0.2, random_state=8, stratify=y_recommended)

X_text_train.shape, X_text_val.shape, X_extra_train.shape, X_extra_val.shape, y_rat_train.shape, y_rat_val.shape, y_rec_train.shape, y_rec_val.shape

In [ ]:
# fix: sparse_categorical_crossentropy отримує мітки 1–5, тоді як Dense(5, activation="softmax") має виходи з індексами 0–4.
#  Треба привести мітки до діапазону 0–4 перед навчанням.

y_rat_train_0idx = np.array(y_rat_train) - 1   # 1–5  до 0–4
y_rat_val_0idx = np.array(y_rat_val) - 1

In [ ]:
# Тепер формуємо шари моделей
# Sequential не підходить для нашого випадку, бо у нас два входи: text_input і total_input
# так як для двох випадків всі шари крім останнього будуть однакові, то можна створити функцію

# Покращення: output_dim=256 - більший простір представлення для слів може покращити результат на текстових задачах

def build_model_bilstm(output_type): #   output_type: "binary" — для Recommended, "multiclass" — для Rating
    # перший вхід: текстова послідовність
    text_input = Input(shape=(MAX_LEN_REVIEW,), name="text_input")
    x = Embedding(input_dim=MAX_WORDS + 1, output_dim=256)(text_input) # Embedding: кожне слово-індекс → вектор із 128 чисел
    x = Bidirectional(LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2))(x)
    # Перший BiLSTM: читає послідовність вперед і назад
    # return_sequences=True — передає всі проміжні стани далі
    x = Dropout(0.3)(x)
    x = Bidirectional(LSTM(32, dropout=0.2, recurrent_dropout=0.2))(x)
    # Другий BiLSTM: стискає всю послідовність в один вектор
    # return_sequences=False за замовчуванням
    x = Dropout(0.3)(x)
    
    # другий вхід: числові + категоріальні ознаки
    extra_input = Input(shape=(X_extra_train.shape[1],), name="extra_input")
    extra_x = Dense(16, activation="relu")(extra_input)
    # об'єднання 
    combined = Concatenate()([x, extra_x])
    combined = Dense(64, activation="relu")(combined)
    combined = Dropout(0.3)(combined)

    # вихідний шар залежить від задач
    if output_type == "binary":
        output = Dense(1, activation="sigmoid", name="output")(combined)  # sigmoid: на виході число від 0 до 1 (ймовірність класу 1)
       
    else:
        output = Dense(5, activation="softmax", name="output")(combined)  # softmax: на виході 5 ймовірностей, сума = 1
       

    model = Model(inputs=[text_input, extra_input], outputs=output)
    return model

In [ ]:
# Компіляція

# Recommended
# model_lstm_rec = build_model_bilstm(output_type="binary")

# model_lstm_rec.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
#     loss="binary_crossentropy",
#     metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
# )

# # Rating
# model_lstm_rat = build_model_bilstm(output_type="multiclass")

# model_lstm_rat.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
#     loss="sparse_categorical_crossentropy", # sparse — бо мітки це цілі числа (0,1,2,3,4), не one-hot вектори
#     metrics=["accuracy"]
# )

In [ ]:
# Так як є значний дисбаланс класів, то треба визначити ваги окремо для кожної задачі. 

# Recommended (бінарна)
# cw_rec = compute_class_weight("balanced", classes=np.unique(y_rec_train), y=y_rec_train)
# class_weight_rec = dict(enumerate(cw_rec))
# print("Recommended weights:", class_weight_rec) # {0: 2.8, 1: 0.6} Клас 0 ("не рекомендує") отримує більшу вагу, бо він рідкісний

# # Rating (5 класів) 
# # Після зсуву міток до 0–4
# cw_rat = compute_class_weight("balanced", classes=np.arange(5), y=y_rat_train_0idx)
# class_weight_rat = dict(enumerate(cw_rat))  # {0: w0, 1: w1, ..., 4: w4} 
# print("Rating weights:", class_weight_rat)   # Клас 0 (оцінка 1) отримає найбільшу вагу, клас 4 (оцінка 5) — найменшу


In [ ]:
# Список колбеків

# callbacks = [
#         EarlyStopping(monitor="val_loss", patience=3,restore_best_weights=True),
#         ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5)]

# Треба зробити функцію замість списку так, як EarlyStopping зберігає внутрішній стан після першого навчання (лічильник епох, найкращий val_loss).
# Друга модель отримує вже "спрацьований" callback

def get_callbacks():
    return [
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5)
    ]

In [ ]:
#  Навчання Recommended

# history_rec = model_lstm_rec.fit(
#     x=[X_text_train, X_extra_train],
#     y=y_rec_train,
#     validation_data=([X_text_val, X_extra_val], y_rec_val),
#     batch_size=128,  # batch_size=128  # швидше, але може погіршити val_loss
#     epochs=20,
#     class_weight=class_weight_rec,
#     callbacks=get_callbacks()
# )

In [ ]:
#  Навчання Rating 

# history_rat = model_lstm_rat.fit(
#     x=[X_text_train, X_extra_train],
#     y=y_rat_train_0idx,     # 0–4
#     validation_data=([X_text_val, X_extra_val], y_rat_val_0idx),
#     batch_size=128,
#     epochs=20,
#     class_weight=class_weight_rat,
#     callbacks=get_callbacks()
# )

In [ ]:
# підбираємо поріг по F1 на validation

# thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
# pred_proba_val = model_lstm_rec.predict([X_text_val, X_extra_val]).squeeze()

# pred_proba_val[:0]

# for t in thresholds:
#     preds = (pred_proba_val >= t).astype(int)
#     f1 = f1_score(y_rec_val, preds)
#     print(f"Threshold {t:.1f} → F1: {f1:.3f}")  # Обираємо поріг з найкращим F1

In [ ]:
# X_text_train, X_text_val,
 # X_extra_train, X_extra_val,
 # y_rat_train, y_rat_val,
 # y_rec_train,y_rec_val


# Передбачення для Recommended

# pred_rec_proba = model_lstm_rec.predict([X_test_pad, X_total_test])   # наприклад [[0.87], [0.23], ....]

# # pred_recommended = (pred_rec_proba.squeeze() >= 0.5).astype(int) # squeeze() прибирає зайвий вимір: (N,1) → (N,)
#                                                                 # >= 0.5 якщо ймовірність більша за 0.5 → клас 1

# pred_recommended = (pred_rec_proba.squeeze() >= 0.3).astype(int)   # обчислений вище поріг 0.3
#                                                                     # якщо ймовірність більша за 0.3 → клас 1

# # Передбачення для Rating 

# pred_rat_proba = model_lstm_rat.predict([X_test_pad, X_total_test])  # масив із 5 ймовірностей для кожного зразка

# pred_rating = np.argmax(pred_rat_proba, axis=1) + 1  # argmax: індекс найбільшої ймовірності (0–4)
#                                                      # +1: повертаємо до оригінальної шкали (1–5)

# # submission
# submission = pd.DataFrame({
#     "Id": df_test_clean["Id"].values,
#     "Rating": pred_rating,
#     "Recommended": pred_recommended
# })

# submission.to_csv("/kaggle/working/submission.csv", index=False)
# print(submission.head(10))

In [ ]:
# X_text_train, X_text_val,
 # X_extra_train, X_extra_val,
 # y_rat_train, y_rat_val,
 # y_rec_train, y_rec_val

# Spearman's Rank Correlation Coefficient вимірює наскільки добре порядок передбачень збігається з порядком істинних значень. 
# Не важливо яке точно число передбачила модель — важливо чи правильно вона ранжує.


# pred_rec_val = model_lstm_rec.predict([X_text_val, X_extra_val]).flatten().round()  # бінарний вихід — округлюємо до 0 або 1

# pred_rating_val = model_lstm_rat.predict([X_text_val, X_extra_val]).argmax(axis=1) + 1  # +1 бо класи 1–5, а індекси 0–4

# spearman_rec = spearmanr(y_rec_val, pred_rec_val).statistic
# spearman_rating = spearmanr(y_rat_val, pred_rating_val).statistic
# mean_spearman  = (spearman_rec + spearman_rating) / 2

# print(f"Spearman Recommended:{spearman_rec:.3f}")
# print(f"Spearman Rating:{spearman_rating:.3f}")
# print(f"Mean Spearman (фінальна метрика): {mean_spearman:.3f}")

**Spearman Recommended: 0.620** — модель непогано ранжує відгуки за ознакою "рекомендую/не рекомендую". Порядок передбачень збігається з реальним у 62% випадків відносно ідеалу.

**Spearman Rating: 0.711**— модель краще справляється з рейтингом. Логічно, бо рейтинг (1–5) має чітку порядкову структуру, яку BiLSTM добре вловлює через контекст тексту.

**Mean Spearman: 0.666**— фінальна метрика конкурсу.

**Висновок:** результат хороший для задачі аналізу тексту без претренованих ембедингів (BERT, GloVe). Простір для покращення є — насамперед через використання претренованих векторів слів.

In [ ]:
# # Порівняння з LogisticRegression + BoW

# pred_rec_lr = model_rec.predict(X_val)
# pred_rating_lr = model_rating.predict(X_val)

# spearman_rec_lr = spearmanr(y_recommended_val, pred_rec_lr).statistic
# spearman_rating_lr = spearmanr(y_rating_val, pred_rating_lr).statistic
# mean_spearman_lr = (spearman_rec_lr + spearman_rating_lr) / 2

# print(f"LR Mean Spearman: {mean_spearman_lr:.3f}")
# print(f"BiLSTM Mean Spearman: 0.666")